# Fine-tune PaddleOCR Devanagari recognizer on handwriting
**Follow each section in order. Run the checkpoint cell after every step. Stop and report any error — do not skip.**

Upload these files to Google Drive **before** starting:
- `My Drive/ocr_imp.zip` — IIIT-HW-Hindi training data
- `My Drive/eval_forms.zip` — 9 real-form test crops (a1.png/a1.txt … a9.png/a9.txt)

## Section 1 · GPU + Drive Setup

In [ ]:
# Cell 1: GPU check — must show Tesla T4
!nvidia-smi

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3: Copy and unzip training data
!cp /content/drive/MyDrive/ocr_imp.zip /content/
!unzip -q /content/ocr_imp.zip -d /content/data
print('Unzip done')

In [ ]:
# Cell 3b: Verify data paths
import os, glob, random

hits = glob.glob('/content/data/**/paddle_labels/rec_gt_train.txt', recursive=True)
assert hits, 'ERROR: rec_gt_train.txt not found under /content/data'
lab = hits[0]
DATA_DIR = os.path.dirname(os.path.dirname(lab))  # parent of paddle_labels/
LABEL_DIR = os.path.join(DATA_DIR, 'paddle_labels')

print('label file :', lab)
print('DATA_DIR   :', DATA_DIR)

lines = open(lab, encoding='utf-8').read().splitlines()
print(f'{len(lines)} train lines (expect 69853)')

sample = random.sample(lines, 500)
bad = [l for l in sample if not os.path.exists(os.path.join(DATA_DIR, l.split('\t')[0]))]
print(f'Missing among 500 sampled: {len(bad)} (expect 0)')
if bad:
    print('SAMPLE BAD PATH:', bad[0])
    raise RuntimeError('Image paths are wrong — fix DATA_DIR before continuing.')

# Make DATA_DIR available to all subsequent cells
import builtins
builtins.DATA_DIR   = DATA_DIR
builtins.LABEL_DIR  = LABEL_DIR
print('✓ Data looks good')

In [ ]:
# Cell 3c: Unzip eval_forms
!cp /content/drive/MyDrive/eval_forms.zip /content/
!unzip -q /content/eval_forms.zip -d /content/eval_forms
import glob
pngs = glob.glob('/content/eval_forms/test/*.png')
print(f'{len(pngs)} eval crops (expect 9)')

## Section 2 · Clone repo + Install Paddle

In [ ]:
# Cell 4: Clone PaddleOCR repo (skip if already done)
import os
if not os.path.exists('/content/PaddleOCR'):
    !git clone https://github.com/PaddlePaddle/PaddleOCR.git
else:
    print('Repo already exists — skipping clone')

In [ ]:
# Cell 5: Install GPU Paddle + dependencies
# This takes ~5 minutes. Colab may ask to restart the kernel — restart if asked,
# then re-run Cell 2 (Drive mount) and Cell 3b (data verify) and continue.
!python -m pip install paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/ -q
!cd /content/PaddleOCR && python -m pip install -r requirements.txt -q
!python -m pip install jiwer pyyaml paddleocr -q
print('Install done')

In [ ]:
# Cell 5b: Checkpoint — Paddle GPU check
import paddle
paddle.utils.run_check()
print('Device:', paddle.device.get_device())
# Expect: PaddlePaddle works well on 1 GPU, device = gpu:0

## Section 3 · Read config and locate weights

In [ ]:
# Cell 6: Print the config so we can read it
CONFIG = '/content/PaddleOCR/configs/rec/PP-OCRv5/multi_language/devanagari_PP-OCRv5_mobile_rec.yaml'
!cat {CONFIG}

In [ ]:
# Cell 7: Parse key values from config
import yaml, os

CONFIG = '/content/PaddleOCR/configs/rec/PP-OCRv5/multi_language/devanagari_PP-OCRv5_mobile_rec.yaml'
with open(CONFIG, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

g = cfg.get('Global', {})
print('pretrained_model  :', g.get('pretrained_model'))
print('character_dict_path:', g.get('character_dict_path'))
print('use_space_char    :', g.get('use_space_char'))
print('max_text_length   :', g.get('max_text_length'))
print('epoch_num         :', g.get('epoch_num'))
print('save_model_dir    :', g.get('save_model_dir'))

# Check dictionary
dict_path = g.get('character_dict_path', '')
full_dict = os.path.join('/content/PaddleOCR', dict_path) if not dict_path.startswith('/') else dict_path
if os.path.exists(full_dict):
    entries = open(full_dict, encoding='utf-8').read().splitlines()
    print(f'\nDictionary: {full_dict}')
    print(f'  entries: {len(entries)} (expect ~568)')
    if not (500 < len(entries) < 650):
        raise RuntimeError(f'Unexpected dictionary size {len(entries)} — stop and report.')
    print('  ✓ dictionary looks right')
else:
    raise FileNotFoundError(f'Dictionary not found: {full_dict}')

## Section 4 · Download pretrained weights

In [ ]:
# Cell 8: Download pretrained .pdparams
# PaddleOCR PP-OCRv5 Devanagari mobile_rec official weights
import os, urllib.request

os.makedirs('/content/pretrained', exist_ok=True)
DST = '/content/pretrained/devanagari_pretrained.pdparams'

# Read pretrained_model from config
import yaml
cfg = yaml.safe_load(open('/content/PaddleOCR/configs/rec/PP-OCRv5/multi_language/devanagari_PP-OCRv5_mobile_rec.yaml', encoding='utf-8'))
url = cfg['Global']['pretrained_model']

if isinstance(url, str) and url.startswith('http'):
    print(f'Downloading from: {url}')
    urllib.request.urlretrieve(url, DST)
    print(f'Downloaded: {os.path.getsize(DST):,} bytes')
else:
    # Config may reference a local path — check if it exists
    local = url if url.startswith('/') else f'/content/PaddleOCR/{url}'
    if os.path.exists(local + '.pdparams'):
        import shutil
        shutil.copy(local + '.pdparams', DST)
        print(f'Copied from local: {os.path.getsize(DST):,} bytes')
    else:
        print(f'pretrained_model value in config: {url!r}')
        print('NOT a URL and file not found locally.')
        print('STOP — look up the official download URL for devanagari_PP-OCRv5_mobile_rec on:')
        print('  https://github.com/PaddlePaddle/PaddleOCR/blob/main/docs/model_list.md')
        raise RuntimeError('Cannot find pretrained weights — manual step required.')

# Checkpoint
size = os.path.getsize(DST)
assert size > 1_000_000, f'File too small ({size} bytes) — download may have failed'
print(f'✓ Pretrained weights ready: {DST} ({size/1e6:.1f} MB)')

## Section 5 · Section 7 data prep (recommended before training)

In [ ]:
# Cell 9: Generate synthetic Hindi line images (stitch 3-6 word images from same writer)
import os, random, glob
import numpy as np
from PIL import Image

# Restore DATA_DIR if session was restarted
hits = glob.glob('/content/data/**/paddle_labels/rec_gt_train.txt', recursive=True)
DATA_DIR  = os.path.dirname(os.path.dirname(hits[0]))
LABEL_DIR = os.path.join(DATA_DIR, 'paddle_labels')

SYNTH_DIR   = '/content/data/synth_lines'
SYNTH_LABEL = '/content/data/synth_labels.txt'
os.makedirs(SYNTH_DIR, exist_ok=True)

# Load train labels
train_lines = open(os.path.join(LABEL_DIR, 'rec_gt_train.txt'), encoding='utf-8').read().splitlines()
entries = []
for l in train_lines:
    parts = l.split('\t')
    if len(parts) == 2:
        entries.append((parts[0], parts[1]))

# Group by writer (top-level subfolder of path)
writers = {}
for path, label in entries:
    writer = path.split('/')[0] if '/' in path else path.split(os.sep)[0]
    writers.setdefault(writer, []).append((path, label))

print(f'Writers found: {len(writers)}')
print('Sample writers:', list(writers.keys())[:5])

TARGET   = 5000  # synthetic line images to generate
MIN_W, MAX_W = 3, 6  # words per line
TARGET_H = 64
GAP_PX   = 4

synth_labels = []
generated = 0
writer_list = [w for w, items in writers.items() if len(items) >= MIN_W]

for i in range(TARGET * 3):  # attempt more than needed
    if generated >= TARGET:
        break
    writer = random.choice(writer_list)
    pool   = writers[writer]
    n      = random.randint(MIN_W, min(MAX_W, len(pool)))
    sample = random.sample(pool, n)

    imgs, labels = [], []
    ok = True
    for path, lbl in sample:
        full = os.path.join(DATA_DIR, path)
        if not os.path.exists(full):
            ok = False; break
        try:
            img = Image.open(full).convert('RGB')
            if img.width < 4 or img.height < 4:
                ok = False; break
            imgs.append(img); labels.append(lbl)
        except Exception:
            ok = False; break
    if not ok or not imgs:
        continue

    # Resize all to same height
    resized = []
    for img in imgs:
        ratio = TARGET_H / img.height
        w = max(1, int(img.width * ratio))
        resized.append(img.resize((w, TARGET_H), Image.LANCZOS))

    # Determine background colour (corner median)
    corner = np.array(resized[0])
    bg = tuple(int(x) for x in np.median(corner[:8, :8].reshape(-1, 3), axis=0))

    total_w = sum(r.width for r in resized) + GAP_PX * (len(resized) - 1)
    canvas  = Image.new('RGB', (total_w, TARGET_H), bg)
    x = 0
    for r in resized:
        canvas.paste(r, (x, 0))
        x += r.width + GAP_PX

    out_name = f'synth_{generated:05d}.jpg'
    out_path = os.path.join(SYNTH_DIR, out_name)
    canvas.save(out_path, quality=90)
    line_label = ' '.join(labels)
    synth_labels.append(f'synth_lines/{out_name}\t{line_label}')
    generated += 1

with open(SYNTH_LABEL, 'w', encoding='utf-8') as f:
    f.write('\n'.join(synth_labels) + '\n')

print(f'✓ Generated {generated} synthetic line images → {SYNTH_LABEL}')

In [ ]:
# Cell 10: Generate English/digit samples with handwriting-style fonts
import os, random, glob
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageFilter

ENG_DIR   = '/content/data/eng_samples'
ENG_LABEL = '/content/data/eng_labels.txt'
os.makedirs(ENG_DIR, exist_ok=True)

# Try to find handwriting-like fonts on the system, fall back to default
FONT_SEARCH = [
    '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
    '/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf',
    '/usr/share/fonts/truetype/freefont/FreeSans.ttf',
]
font_path = next((p for p in FONT_SEARCH if os.path.exists(p)), None)

# Download a handwriting font if none found
if font_path is None:
    !pip install fonttools -q
    !wget -q -O /tmp/handwriting.ttf \
        'https://github.com/google/fonts/raw/main/ofl/caveat/Caveat-Regular.ttf'
    font_path = '/tmp/handwriting.ttf' if os.path.exists('/tmp/handwriting.ttf') else None

print(f'Font: {font_path}')

# Text samples: English phrases + digit strings common in forms
TEXTS = [
    'Name', 'Date', 'Yes', 'No', 'Signature', 'Village', 'District',
    'Male', 'Female', 'Yes No', 'Please tick', 'NA', 'Not applicable',
    'Good', 'Average', 'Poor', 'Excellent', 'Total', 'Page 1', 'Page 2',
    '1 2 3 4 5 6 7 8 9 10', '01/01/2024', '15/08/2025', '2024', '2025',
    '9876543210', '1234567890', '100', '500', '1000', '75%', '50%',
    'Q1 Q2 Q3 Q4', 'Section A', 'Part B', 'Yes / No',
    '1', '2', '3', '4', '5', '6', '7', '8', '9', '10',
    'a b c d e', 'Monday Tuesday', 'January February March',
    'Please fill in the form', 'Thank you', 'Signature here',
    'Roll No', 'Sr No', 'Gram Panchayat', 'Taluka', 'Maharashtra',
]

TARGET_H  = 64
N_SAMPLES = 2000
eng_labels_out = []

for i in range(N_SAMPLES):
    text  = random.choice(TEXTS)
    fsize = random.randint(28, 48)

    try:
        font = ImageFont.truetype(font_path, fsize) if font_path else ImageFont.load_default()
    except Exception:
        font = ImageFont.load_default()

    # Measure text
    dummy = Image.new('RGB', (1, 1))
    d = ImageDraw.Draw(dummy)
    bbox = d.textbbox((0, 0), text, font=font)
    tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
    pad = random.randint(4, 12)
    W, H = tw + 2 * pad, th + 2 * pad
    if W < 8 or H < 8:
        continue

    # Background: off-white with slight noise
    bg_val = random.randint(220, 255)
    img    = Image.new('RGB', (W, H), (bg_val, bg_val, bg_val))
    arr    = np.array(img, dtype=np.int16)
    noise  = np.random.randint(-10, 10, arr.shape, dtype=np.int16)
    arr    = np.clip(arr + noise, 0, 255).astype(np.uint8)
    img    = Image.fromarray(arr)

    # Text colour: dark
    ink = tuple(random.randint(0, 50) for _ in range(3))
    draw = ImageDraw.Draw(img)
    draw.text((pad, pad), text, font=font, fill=ink)

    # Augment: slight blur, brightness
    if random.random() < 0.4:
        img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.3, 0.8)))
    from PIL import ImageEnhance
    img = ImageEnhance.Brightness(img).enhance(random.uniform(0.85, 1.1))
    img = ImageEnhance.Contrast(img).enhance(random.uniform(0.8, 1.2))

    # Slight rotation
    angle = random.uniform(-3, 3)
    img   = img.rotate(angle, expand=True, fillcolor=(bg_val, bg_val, bg_val))

    # Resize to target height
    ratio = TARGET_H / img.height
    img   = img.resize((max(1, int(img.width * ratio)), TARGET_H), Image.LANCZOS)

    out_name = f'eng_{i:05d}.jpg'
    img.save(os.path.join(ENG_DIR, out_name), quality=90)
    eng_labels_out.append(f'eng_samples/{out_name}\t{text}')

with open(ENG_LABEL, 'w', encoding='utf-8') as f:
    f.write('\n'.join(eng_labels_out) + '\n')

print(f'✓ Generated {len(eng_labels_out)} English/digit samples → {ENG_LABEL}')

### Check max label length across all files
If it exceeds `max_text_length` in the config we need to override it.

In [ ]:
# Cell 11: Find maximum label length across all label files
import glob as _glob

all_label_files = [
    '/content/data/paddle_labels/rec_gt_train.txt',
    '/content/data/synth_labels.txt',
    '/content/data/eng_labels.txt',
]
# Update path if DATA_DIR differs
hits = _glob.glob('/content/data/**/paddle_labels/rec_gt_train.txt', recursive=True)
if hits:
    all_label_files[0] = hits[0]

max_len = 0
for lf in all_label_files:
    if not os.path.exists(lf):
        print(f'SKIP (not found): {lf}'); continue
    for line in open(lf, encoding='utf-8'):
        parts = line.strip().split('\t')
        if len(parts) == 2:
            max_len = max(max_len, len(parts[1]))

import yaml
cfg = yaml.safe_load(open('/content/PaddleOCR/configs/rec/PP-OCRv5/multi_language/devanagari_PP-OCRv5_mobile_rec.yaml', encoding='utf-8'))
config_max = cfg['Global']['max_text_length']
print(f'Longest label in data : {max_len}')
print(f'Config max_text_length: {config_max}')
MAX_TEXT_LEN = max(max_len + 5, config_max)
print(f'Will use max_text_length={MAX_TEXT_LEN} (override on command line if > config)')

## Section 6 · Smoke test (1 epoch, 2000-sample subset)

In [ ]:
# Cell 12: Create small subsets for smoke test
import os, random, glob as _glob

hits = _glob.glob('/content/data/**/paddle_labels/rec_gt_train.txt', recursive=True)
DATA_DIR  = os.path.dirname(os.path.dirname(hits[0]))
LABEL_DIR = os.path.join(DATA_DIR, 'paddle_labels')

train_lines = open(os.path.join(LABEL_DIR, 'rec_gt_train.txt'), encoding='utf-8').read().splitlines()
val_lines   = open(os.path.join(LABEL_DIR, 'rec_gt_val.txt'),   encoding='utf-8').read().splitlines()

random.shuffle(train_lines)

SMOKE_DIR = '/content/smoke_labels'
os.makedirs(SMOKE_DIR, exist_ok=True)

with open(f'{SMOKE_DIR}/smoke_train.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(train_lines[:2000]) + '\n')
with open(f'{SMOKE_DIR}/smoke_val.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(val_lines[:500]) + '\n')

print(f'Smoke train: 2000 lines → {SMOKE_DIR}/smoke_train.txt')
print(f'Smoke val  :  500 lines → {SMOKE_DIR}/smoke_val.txt')
print(f'DATA_DIR   : {DATA_DIR}')

In [ ]:
# Cell 13: Run smoke test — 1 epoch
# Read DATA_DIR
import glob as _gl, os
hits = _gl.glob('/content/data/**/paddle_labels/rec_gt_train.txt', recursive=True)
DATA_DIR = os.path.dirname(os.path.dirname(hits[0]))

CONFIG = '/content/PaddleOCR/configs/rec/PP-OCRv5/multi_language/devanagari_PP-OCRv5_mobile_rec.yaml'

%cd /content/PaddleOCR
!python tools/train.py \
    -c {CONFIG} \
    -o Global.pretrained_model=/content/pretrained/devanagari_pretrained \
       Global.save_model_dir=/content/drive/MyDrive/ocr_out/smoke \
       Global.epoch_num=1 \
       Global.eval_batch_step=[0,200] \
       Train.dataset.data_dir={DATA_DIR} \
       "Train.dataset.label_file_list=[/content/smoke_labels/smoke_train.txt]" \
       Eval.dataset.data_dir={DATA_DIR} \
       "Eval.dataset.label_file_list=[/content/smoke_labels/smoke_val.txt]" \
       Train.loader.batch_size_per_card=64 \
       2>&1 | tail -80

**Checkpoint after smoke test:**
- Loss prints and goes **down** ✓
- Eval accuracy prints ✓  
- No OOM error ✓ (if OOM → lower `batch_size_per_card` to 32)
- A checkpoint file appears in `MyDrive/ocr_out/smoke/` ✓
- Note **seconds/iter** from the log — multiply by total iters to estimate full-run time

## Section 7 · Full training run

In [ ]:
# Cell 14: FULL training
# Adjust EPOCHS and BATCH_SIZE if needed based on smoke-test timing
import glob as _gl, os
hits = _gl.glob('/content/data/**/paddle_labels/rec_gt_train.txt', recursive=True)
DATA_DIR  = os.path.dirname(os.path.dirname(hits[0]))
LABEL_DIR = os.path.join(DATA_DIR, 'paddle_labels')

CONFIG     = '/content/PaddleOCR/configs/rec/PP-OCRv5/multi_language/devanagari_PP-OCRv5_mobile_rec.yaml'
EPOCHS     = 20
BATCH_SIZE = 64    # lower to 32 if OOM
LR         = 0.0002  # ~1/5 of typical default 0.001

# Merge all label files into one combined train list
import yaml
TRAIN_LABELS = [
    os.path.join(LABEL_DIR, 'rec_gt_train.txt'),
    '/content/data/synth_labels.txt',
    '/content/data/eng_labels.txt',
]
VAL_LABEL = os.path.join(LABEL_DIR, 'rec_gt_val.txt')

# Build label_file_list string for -o override
train_list_str = '[' + ','.join(TRAIN_LABELS) + ']'

%cd /content/PaddleOCR
!python tools/train.py \
    -c {CONFIG} \
    -o Global.pretrained_model=/content/pretrained/devanagari_pretrained \
       Global.save_model_dir=/content/drive/MyDrive/ocr_out/devanagari_hw \
       Global.epoch_num={EPOCHS} \
       Global.eval_batch_step=[0,2000] \
       Global.use_space_char=True \
       "Train.dataset.label_file_list={train_list_str}" \
       Train.dataset.data_dir={DATA_DIR} \
       "Eval.dataset.label_file_list=[{VAL_LABEL}]" \
       Eval.dataset.data_dir={DATA_DIR} \
       Train.loader.batch_size_per_card={BATCH_SIZE} \
       Optimizer.lr.learning_rate={LR}

In [ ]:
# Cell 14b: Resume after a disconnect (run instead of Cell 14 if session was lost)
import glob as _gl, os
hits = _gl.glob('/content/data/**/paddle_labels/rec_gt_train.txt', recursive=True)
DATA_DIR  = os.path.dirname(os.path.dirname(hits[0]))
LABEL_DIR = os.path.join(DATA_DIR, 'paddle_labels')

CONFIG     = '/content/PaddleOCR/configs/rec/PP-OCRv5/multi_language/devanagari_PP-OCRv5_mobile_rec.yaml'
EPOCHS     = 20
BATCH_SIZE = 64
LR         = 0.0002

TRAIN_LABELS = [
    os.path.join(LABEL_DIR, 'rec_gt_train.txt'),
    '/content/data/synth_labels.txt',
    '/content/data/eng_labels.txt',
]
VAL_LABEL = os.path.join(LABEL_DIR, 'rec_gt_val.txt')
train_list_str = '[' + ','.join(TRAIN_LABELS) + ']'

%cd /content/PaddleOCR
!python tools/train.py \
    -c {CONFIG} \
    -o Global.checkpoints=/content/drive/MyDrive/ocr_out/devanagari_hw/latest \
       Global.save_model_dir=/content/drive/MyDrive/ocr_out/devanagari_hw \
       Global.epoch_num={EPOCHS} \
       Global.eval_batch_step=[0,2000] \
       Global.use_space_char=True \
       "Train.dataset.label_file_list={train_list_str}" \
       Train.dataset.data_dir={DATA_DIR} \
       "Eval.dataset.label_file_list=[{VAL_LABEL}]" \
       Eval.dataset.data_dir={DATA_DIR} \
       Train.loader.batch_size_per_card={BATCH_SIZE} \
       Optimizer.lr.learning_rate={LR}

## Section 8 · Export best checkpoint to inference model

In [ ]:
# Cell 15: Export to inference model
CONFIG = '/content/PaddleOCR/configs/rec/PP-OCRv5/multi_language/devanagari_PP-OCRv5_mobile_rec.yaml'

%cd /content/PaddleOCR
!python tools/export_model.py \
    -c {CONFIG} \
    -o Global.pretrained_model=/content/drive/MyDrive/ocr_out/devanagari_hw/best_accuracy \
       Global.save_inference_dir=/content/drive/MyDrive/ocr_out/inference

import os, glob
exported = glob.glob('/content/drive/MyDrive/ocr_out/inference/*')
print('Exported files:', exported)
# Must include model.pdmodel, model.pdiparams, inference.yml (or similar)

In [ ]:
# Cell 15b: Quick verify export loads
from paddleocr import PaddleOCR
ocr_ft = PaddleOCR(
    text_recognition_model_name='devanagari_PP-OCRv5_mobile_rec',
    text_recognition_model_dir='/content/drive/MyDrive/ocr_out/inference',
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
)
test_crop = '/content/eval_forms/test/a1.png'
result = ocr_ft.predict(test_crop)
print('Test predict result:', result)
print('✓ Export verified')

## Section 9 · Evaluate on real-form crops (the test that counts)

In [ ]:
# Cell 16: CER evaluation — baseline vs fine-tuned
import glob, os, jiwer
from paddleocr import PaddleOCR

def norm(s):
    return ' '.join(s.split())

def make(model_dir=None):
    kw = dict(
        text_recognition_model_name='devanagari_PP-OCRv5_mobile_rec',
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
    )
    if model_dir:
        kw['text_recognition_model_dir'] = model_dir
    return PaddleOCR(**kw)

def read(ocr, path):
    items = []
    for r in ocr.predict(path):
        # Adapt if key names differ — print r if this fails
        try:
            rec_texts = r['rec_texts']
            rec_boxes = r['rec_boxes']
        except (KeyError, TypeError):
            print('Unexpected result structure:', r)
            return ''
        for text, box in zip(rec_texts, rec_boxes):
            x1, y1, x2, y2 = [int(v) for v in box]
            items.append((text, x1, (y1 + y2) / 2, y2 - y1))
    if not items:
        return ''
    items.sort(key=lambda t: t[2])
    tol = sorted(t[3] for t in items)[len(items) // 2] * 0.6
    lines, cur = [], [items[0]]
    for it in items[1:]:
        if abs(it[2] - sum(c[2] for c in cur) / len(cur)) <= tol:
            cur.append(it)
        else:
            lines.append(cur); cur = [it]
    lines.append(cur)
    return ' '.join(t[0] for ln in lines for t in sorted(ln, key=lambda t: t[1]))

def cer_report(ocr, folder, out_file):
    scores = []
    hindi_scores, eng_scores = [], []
    with open(out_file, 'w', encoding='utf-8') as f:
        for img in sorted(glob.glob(folder + '/*.png')):
            truth_path = os.path.splitext(img)[0] + '.txt'
            if not os.path.exists(truth_path):
                print(f'WARNING: no ground truth for {img}'); continue
            truth = norm(open(truth_path, encoding='utf-8').read())
            pred  = norm(read(ocr, img))
            e = jiwer.cer(truth, pred)
            scores.append(e)
            # Heuristic: if >40% chars are Devanagari → Hindi-heavy
            deva_count = sum(1 for c in truth if '\u0900' <= c <= '\u097F')
            is_hindi   = deva_count / max(1, len(truth.replace(' ', ''))) > 0.4
            if is_hindi:
                hindi_scores.append(e)
            else:
                eng_scores.append(e)
            f.write(f'{os.path.basename(img)}\n')
            f.write(f'  correct : {truth}\n')
            f.write(f'  model   : {pred}\n')
            f.write(f'  CER     : {e:.3f}  [{"Hindi" if is_hindi else "English"}]\n\n')
    mean = sum(scores) / len(scores) if scores else float('nan')
    f_obj = open(out_file, 'a', encoding='utf-8')
    f_obj.write(f'\n=== SUMMARY ===\n')
    f_obj.write(f'Overall CER : {mean:.3f} ({len(scores)} crops)\n')
    f_obj.write(f'Hindi-heavy : {sum(hindi_scores)/len(hindi_scores):.3f} ({len(hindi_scores)} crops)\n' if hindi_scores else 'Hindi-heavy : —\n')
    f_obj.write(f'Eng-heavy   : {sum(eng_scores)/len(eng_scores):.3f} ({len(eng_scores)} crops)\n' if eng_scores else 'Eng-heavy   : —\n')
    f_obj.close()
    return mean, hindi_scores, eng_scores

print('--- Baseline (pretrained, no fine-tuning) ---')
b_all, b_hi, b_en = cer_report(make(), '/content/eval_forms/test', '/content/drive/MyDrive/ocr_out/baseline.txt')
print(f'Baseline CER  all={b_all:.3f}  hindi={sum(b_hi)/len(b_hi):.3f if b_hi else "—"}  eng={sum(b_en)/len(b_en):.3f if b_en else "—"}')

print('\n--- Fine-tuned ---')
f_all, f_hi, f_en = cer_report(
    make('/content/drive/MyDrive/ocr_out/inference'),
    '/content/eval_forms/test',
    '/content/drive/MyDrive/ocr_out/finetuned.txt'
)
print(f'Fine-tuned CER  all={f_all:.3f}  hindi={sum(f_hi)/len(f_hi):.3f if f_hi else "—"}  eng={sum(f_en)/len(f_en):.3f if f_en else "—"}')

print('\n=== RESULT ===')
delta = f_all - b_all
if delta < 0:
    print(f'✓ IMPROVEMENT: CER {b_all:.3f} → {f_all:.3f} (Δ={delta:+.3f})')
else:
    print(f'✗ NO IMPROVEMENT: CER {b_all:.3f} → {f_all:.3f} (Δ={delta:+.3f}) — report this honestly')

## Section 10 · Final deliverables checklist

In [ ]:
# Cell 17: List all deliverables in MyDrive/ocr_out
import glob, os

OUT = '/content/drive/MyDrive/ocr_out'
for root, dirs, files in os.walk(OUT):
    level = root.replace(OUT, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in sorted(files):
        size = os.path.getsize(os.path.join(root, f))
        print(f'{indent}  {f}  ({size/1e6:.1f} MB)')

print('\nChecklist:')
items = [
    (f'{OUT}/inference', 'inference/ export folder'),
    (f'{OUT}/devanagari_hw/best_accuracy.pdparams', 'best_accuracy checkpoint'),
    (f'{OUT}/baseline.txt', 'baseline CER report'),
    (f'{OUT}/finetuned.txt', 'fine-tuned CER report'),
]
for path, label in items:
    ok = os.path.exists(path)
    print(f'  [{"✓" if ok else "✗"}] {label}: {path}')

print('\nDownload the inference/ folder to:')
print('  C:\\ocr_project\\models\\devanagari_hw\\')

## Section 11 · Local usage (Windows)

After downloading `inference/` to `C:\ocr_project\models\devanagari_hw\`:

```python
from paddleocr import PaddleOCR

paddle = PaddleOCR(
    text_recognition_model_name='devanagari_PP-OCRv5_mobile_rec',
    text_recognition_model_dir=r'C:\ocr_project\models\devanagari_hw',
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    enable_mkldnn=False,   # avoids the CPU crash seen earlier
)
```

> **Note:** Use this only for the free-text Q14 field.  
> Keep circled 1–10 scale answers and numeric fields on OpenCV/digit-only code.